# Kayfa Student Analytics — Day 2: The 15 Questions
## Data Analytics Evaluation
> **Goal:** Answer all 15 questions with clean charts + written insights + CTAs.
>
> **Order:** Q1–Q10, Q12, Q15 first → then clustering → Q11, Q13, Q14

In [37]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import zipfile, os, json

with zipfile.ZipFile('/content/cleaned_data.zip', 'r') as z:
    z.extractall('/content/cleaned_data/')
for root, dirs, files in os.walk('/content/cleaned_data/'):
    for f in files:
        print(os.path.join(root, f))

/content/cleaned_data/courses.csv
/content/cleaned_data/students_clean.csv
/content/cleaned_data/submissions_clean.csv
/content/cleaned_data/master_student_dataset.csv
/content/cleaned_data/concepts_clean.csv
/content/cleaned_data/grades_clean.csv
/content/cleaned_data/attendance_clean.csv
/content/cleaned_data/groups_clean.csv
/content/cleaned_data/engagement_clean.csv


In [38]:
P = '/content/cleaned_data/'

master_student_dataset = pd.read_csv(f'{P}master_student_dataset.csv')
courses = pd.read_csv(f'{P}courses.csv')
groups = pd.read_csv(f'{P}groups_clean.csv')
students = pd.read_csv(f'{P}students_clean.csv')
grades = pd.read_csv(f'{P}grades_clean.csv', parse_dates=['date'])
attendance = pd.read_csv(f'{P}attendance_clean.csv', parse_dates=['session_datetime'])
concepts = pd.read_csv(f'{P}concepts_clean.csv', parse_dates=['timestamp'])
engagement = pd.read_csv(f'{P}engagement_clean.csv', parse_dates=['event_datetime'])
submissions = pd.read_csv(f'{P}submissions_clean.csv', parse_dates=['deadline', 'submitted_at'])

print('✅ All loaded')
for name, df in [('courses',courses),('groups',groups),('students',students),
    ('grades',grades),('attendance',attendance),('concepts',concepts),
    ('engagement',engagement),('submissions',submissions),('master',master_student_dataset)]:
    print(f'  {name}: {df.shape}')


✅ All loaded
  courses: (7, 8)
  groups: (10, 7)
  students: (500, 8)
  grades: (5502, 9)
  attendance: (13010, 7)
  concepts: (12003, 10)
  engagement: (30857, 6)
  submissions: (1504, 9)
  master: (500, 27)


In [39]:
BLUE = ['#08306b', '#08519c', '#2171b5', '#4292c6', '#6baed6', '#9ecae1', '#c6dbef', '#deebf7']
BG = '#fafbfd'

def style_fig(fig, title=''):
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color='#08306b')),
        plot_bgcolor=BG, paper_bgcolor='white',
        font=dict(family='Segoe UI', size=12, color='#333'),
        margin=dict(l=60, r=30, t=60, b=50),
        showlegend=True
    )
    fig.update_xaxes(gridcolor='#e8e8e8')
    fig.update_yaxes(gridcolor='#e8e8e8')
    return fig

print('\u2705 All data loaded, theme set')


✅ All data loaded, theme set


---
# Q1 · Attendance Rate per Group (4 pts)
> Which groups sit well below the platform average?

In [40]:

att_grp = attendance.groupby('group_id').agg(
    total=('status', 'count'),
    attended=('status', lambda x: (x == 'attended').sum())
).reset_index()
att_grp['rate'] = (att_grp['attended'] / att_grp['total'] * 100).round(1)
att_grp = att_grp.merge(groups[['group_id', 'group_name']], on='group_id')
att_grp = att_grp.sort_values('rate', ascending=True)

platform_avg = att_grp['rate'].mean()

colors = [BLUE[0] if r < platform_avg else BLUE[4] for r in att_grp['rate']]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=att_grp['group_name'], x=att_grp['rate'],
    orientation='h', marker_color=colors,
    text=att_grp['rate'].apply(lambda x: f'{x}%'), textposition='outside'
))
fig.add_vline(x=platform_avg, line_dash='dash', line_color='#e63946',
              annotation_text=f'Avg: {platform_avg:.1f}%', annotation_position='top right')
style_fig(fig, 'Q1: Attendance Rate by Group')
fig.update_layout(yaxis_title='', xaxis_title='Attendance Rate (%)', xaxis_range=[20, 105])
fig.show()

### Lets Dig Even Deeper

#### Which students are driving the low attendance?

In [41]:
target_groups = ['G10', 'G07']

student_att = (
    attendance[attendance['group_id'].isin(target_groups)]
    .groupby('student_id')
    .agg(
        total_sessions=('status', 'count'),
        attended_sessions=('status', lambda x: (x == 'attended').sum())
    )
    .reset_index()
)

student_att['attendance_rate'] = (
    student_att['attended_sessions']
    / student_att['total_sessions']
    * 100
).round(1)

student_att = student_att.merge(
    students[['student_id', 'full_name']],
    on='student_id'
)

student_att.sort_values('attendance_rate').head(15)

,student_id,total_sessions,attended_sessions,attendance_rate,full_name
14,S0140,26,11,42.3,Rowan Mahmoud
22,S0201,26,12,46.2,Rowan ElBaz
27,S0263,26,12,46.2,Aya Ramadan
43,S0412,26,12,46.2,Amr Abdelaziz
16,S0142,26,13,50.0,Habiba Halim
24,S0210,26,13,50.0,Nada Nasr
7,S0073,26,13,50.0,Fady Refaat
0,S0006,26,13,50.0,Aya ElSayed
50,S0453,26,13,50.0,Marwan ElBaz
44,S0414,26,13,50.0,Dina ElMasry


In [42]:
group_course = groups[['group_id', 'group_name', 'course_id']]

att_course = (
    attendance.groupby('group_id')
    .agg(
        total=('status','count'),
        attended=('status', lambda x: (x=='attended').sum())
    )
    .reset_index()
)

att_course['attendance_rate'] = (
    att_course['attended'] /
    att_course['total'] * 100
).round(1)

att_course = att_course.merge(group_course,on='group_id')
att_course = att_course.merge(
    courses[['course_id','course_name']],
    on='course_id'
)

att_course.sort_values(['course_name','attendance_rate'])

,group_id,total,attended,attendance_rate,group_name,course_id,course_name
9,G10,26,17,65.4,Group 10 — C007,C007,Cybersecurity Essentials
1,G02,1456,1149,78.9,Group 02 — C001,C001,Data Analytics Fundamentals
0,G01,1356,1091,80.5,Group 01 — C001,C001,Data Analytics Fundamentals
6,G07,1508,908,60.2,Group 07 — C005,C005,Digital Marketing
7,G08,1561,1238,79.3,Group 08 — C006,C006,Machine Learning Basics
8,G09,1326,1051,79.3,Group 09 — C006,C006,Machine Learning Basics
3,G04,1692,1306,77.2,Group 04 — C002,C002,Python Programming
2,G03,1431,1107,77.4,Group 03 — C002,C002,Python Programming
5,G06,1458,1158,79.4,Group 06 — C004,C004,UI/UX Design
4,G05,1196,966,80.8,Group 05 — C003,C003,Web Development


In [43]:
att_course[att_course['group_id'].isin(['G10','G07'])]

,group_id,total,attended,attendance_rate,group_name,course_id,course_name
6,G07,1508,908,60.2,Group 07 — C005,C005,Digital Marketing
9,G10,26,17,65.4,Group 10 — C007,C007,Cybersecurity Essentials


ARE THESE COURSES HARD? LONG?

In [44]:
course_perf = (
    grades.groupby('course_id')
    .agg(
        avg_score=('score','mean')
    )
    .reset_index()
)

course_perf = course_perf.merge(
    courses[['course_id','course_name','difficulty_level']],
    on='course_id'
)

course_perf.sort_values('avg_score')

,course_id,avg_score,course_name,difficulty_level
4,C005,59.078213,Digital Marketing,Beginner
2,C003,70.270020,Web Development,Intermediate
1,C002,71.532702,Python Programming,Beginner
3,C004,71.631169,UI/UX Design,Beginner
0,C001,72.595328,Data Analytics Fundamentals,Beginner
5,C006,72.812695,Machine Learning Basics,Advanced
6,C007,76.154545,Cybersecurity Essentials,Intermediate


In [45]:
late_rate = (
    submissions.groupby('course_id')
    .agg(
        total_submissions=('submission_id','count'),
        late_submissions=('is_late','sum')
    )
    .reset_index()
)

late_rate['late_pct'] = (
    late_rate['late_submissions']
    / late_rate['total_submissions']
    * 100
).round(1)

late_rate = late_rate.merge(
    courses[['course_id','course_name']],
    on='course_id'
)

late_rate.sort_values('late_pct', ascending=False)

,course_id,total_submissions,late_submissions,late_pct,course_name
4,C005,176,99,56.2,Digital Marketing
1,C002,360,125,34.7,Python Programming
0,C001,325,112,34.5,Data Analytics Fundamentals
2,C003,138,46,33.3,Web Development
5,C006,334,105,31.4,Machine Learning Basics
3,C004,168,51,30.4,UI/UX Design
6,C007,3,0,0.0,Cybersecurity Essentials


In [46]:
course_time = (
    submissions.groupby('course_id')
    .agg(
        avg_time_spent=('time_spent_minutes','mean')
    )
    .reset_index()
)

course_time = course_time.merge(
    courses[['course_id','course_name']],
    on='course_id'
)

course_time.sort_values('avg_time_spent')

,course_id,avg_time_spent,course_name
6,C007,100.333333,Cybersecurity Essentials
3,C004,116.958333,UI/UX Design
2,C003,118.514493,Web Development
4,C005,119.176136,Digital Marketing
1,C002,119.280556,Python Programming
0,C001,120.295385,Data Analytics Fundamentals
5,C006,123.739521,Machine Learning Basics


In [47]:
mastery = (
    concepts.groupby('course_id')
    .agg(
        avg_score=('score_pct','mean')
    )
    .reset_index()
)

mastery = mastery.merge(
    courses[['course_id','course_name']],
    on='course_id'
)

mastery.sort_values('avg_score')

,course_id,avg_score,course_name
4,C005,61.319777,Digital Marketing
1,C002,69.660313,Python Programming
0,C001,71.845644,Data Analytics Fundamentals
6,C007,71.933333,Cybersecurity Essentials
2,C003,72.204167,Web Development
5,C006,72.267342,Machine Learning Basics
3,C004,73.275744,UI/UX Design


Is attendance affecting academic performance?

In [48]:
student_att = (
    attendance.groupby('student_id')
    .agg(
        total=('status','count'),
        attended=('status', lambda x: (x=='attended').sum())
    )
    .reset_index()
)

student_att['attendance_rate'] = (
    student_att['attended']
    / student_att['total']
    * 100
)

In [49]:
student_grade = (
    grades.groupby('student_id')['score']
    .mean()
    .reset_index()
)

student_grade.rename(
    columns={'score':'avg_grade'},
    inplace=True
)

In [50]:
impact = student_att.merge(
    student_grade,
    on='student_id'
)

impact[['attendance_rate','avg_grade']].corr()

,attendance_rate,avg_grade
attendance_rate,1.00000,0.46763
avg_grade,0.46763,1.00000


In [51]:
import plotly.express as px

fig = px.scatter(
    impact,
    x='attendance_rate',
    y='avg_grade',
    hover_data=['student_id'],
    trendline='ols',
    title='Attendance Rate vs Average Grade'
)

fig.show()

Students with stronger attendance generally achieve higher assessment scores. The low attendance observed in Groups 07 and 10 may therefore contribute to weaker academic outcomes and should be addressed as a priority.

In [52]:
eng = engagement.merge(
    students[['student_id','group_id']],
    on='student_id'
)

group_eng = (
    eng.groupby('group_id')
    .agg(
        total_events=('event_id','count'),
        avg_duration=('duration_seconds','mean')
    )
    .reset_index()
)

group_eng.sort_values('avg_duration')

,group_id,total_events,avg_duration
9,G10,34,142.705882
6,G07,3032,153.584103
5,G06,3500,167.194571
4,G05,2754,172.348947
2,G03,3469,173.426924
1,G02,3680,173.638587
8,G09,3259,173.776619
7,G08,3808,173.846639
0,G01,3243,181.105150
3,G04,4078,181.371996


In [53]:
import plotly.express as px

group_eng = group_eng.sort_values('avg_duration')

fig = px.bar(
    group_eng,
    x='group_id',
    y='avg_duration',
    text='avg_duration',
    title='Average Engagement Duration by Group'
)

fig.update_traces(texttemplate='%{text:.1f}s')
fig.update_layout(
    xaxis_title='Group',
    yaxis_title='Average Duration (seconds)'
)

fig.show()

In [54]:
attendance_rates = att_grp[['group_id','rate']]

eng_vs_att = group_eng.merge(
    attendance_rates,
    on='group_id'
)

eng_vs_att

,group_id,total_events,avg_duration,rate
0,G10,34,142.705882,65.4
1,G07,3032,153.584103,60.2
2,G06,3500,167.194571,79.4
3,G05,2754,172.348947,80.8
4,G03,3469,173.426924,77.4
5,G02,3680,173.638587,78.9
6,G09,3259,173.776619,79.3
7,G08,3808,173.846639,79.3
8,G01,3243,181.105150,80.5
9,G04,4078,181.371996,77.2


### Q1 — Insight
Insight

The platform-wide attendance rate is 75.8%.

Two groups are significantly below average:

| Group           | Attendance Rate | Gap vs Avg |
| --------------- | --------------- | ---------- |
| Group 07 — C005 | 60.2%           | -15.6 pp   |
| Group 10 — C007 | 65.4%           | -10.4 pp   |


### Q1 — CTA
Recommended Action (CTA)
Prioritize investigation of Group 07 and Group 10.
Review instructor engagement, scheduling conflicts, and student participation patterns.
Contact students with repeated absences and implement attendance interventions.
If these groups belong to specific courses, review course difficulty and delivery methods.
Implement attendance interventions for Groups G07 and G10, as improving attendance is likely to improve student performance and overall learning outcomes.

Executive Summary
Insight

The attendance problem is primarily driven by Group 07 (Digital Marketing) and Group 10 (Cybersecurity Essentials). However, the root causes differ:

Digital Marketing shows low attendance, low grades, low mastery, and the highest late-submission rate, indicating a broader learning and engagement challenge.
Cybersecurity Essentials shows low attendance but maintains relatively strong academic outcomes, suggesting attendance may not be affecting performance to the same extent and that additional investigation is required due to the small sample size.
CTA
Prioritize Digital Marketing for immediate intervention.
Review course structure, assessments, and engagement strategies in Digital Marketing.
Monitor Cybersecurity Essentials for additional data before making major curriculum changes.
Establish attendance and engagement dashboards to identify struggling groups earlier in future cohorts.

---
# Q2 · Score Distribution by Assessment Type (4 pts)
> Where is performance most volatile?

In [55]:
type_stats = grades.groupby('type')['score'].agg(
    ['mean', 'std', 'median', 'count']
).reset_index()

type_stats.columns = ['type', 'avg_score', 'std', 'median', 'count']
type_stats = type_stats.sort_values('avg_score', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=type_stats['type'],
    x=type_stats['avg_score'],
    orientation='h',
    marker_color=BLUE[2],
    name='Avg Score',
    text=type_stats['avg_score'].round(1),
    textposition='outside',
    error_x=dict(
        type='data',
        array=type_stats['std'].round(1),
        color=BLUE[0]
    )
))

style_fig(fig, 'Q2: Average Score by Assessment Type (± Std Dev)')

fig.update_layout(
    xaxis_title='Average Score',
    yaxis_title='Assessment Type',
    xaxis_range=[0, 115]
)

overall_avg = grades['score'].mean()

fig.add_vline(
    x=overall_avg,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Overall Avg: {overall_avg:.1f}'
)
fig.show()


In [56]:
type_stats[['type','avg_score','median','std','count']]

,type,avg_score,median,std,count
0,assignment,65.316100,65.50,12.860244,1500
3,quiz,72.383700,72.20,11.965612,2000
2,practical,72.406400,72.70,11.545622,1000
1,exam,72.630339,72.75,11.359338,1002


In [57]:
assignment_scores = grades[grades['type'] == 'assignment']

assignment_course = (
    assignment_scores.groupby('course_id')['score']
    .mean()
    .reset_index()
    .merge(courses[['course_id','course_name']], on='course_id')
    .sort_values('score')
)

assignment_course

,course_id,score,course_name
4,C005,53.664368,Digital Marketing
2,C003,65.758696,Web Development
1,C002,65.928056,Python Programming
0,C001,67.233796,Data Analytics Fundamentals
5,C006,67.437538,Machine Learning Basics
6,C007,67.566667,Cybersecurity Essentials
3,C004,67.765476,UI/UX Design


### Q2 — Insight
Insight

Assignment performance is the primary academic weakness across the platform, with students scoring substantially lower on assignments than on exams, quizzes, or practical assessments. This issue is especially severe in Digital Marketing, where assignment scores average only 53.7%, significantly below all other courses. Combined with low attendance, low mastery scores, and the highest late-submission rate, Digital Marketing appears to be the main driver of poor assignment outcomes.

CTA

Review assignment structure within the Digital Marketing course, including workload, grading criteria, instructions, and submission deadlines. Introducing milestone-based assignments, earlier feedback cycles, and targeted academic support may help improve both assignment completion and overall course performance.


---
# Q3 · Highest & Lowest Avg Grade by Course (5 pts)
> How does grade spread differ between courses?

In [58]:

stu_course = students.merge(
    groups[['group_id', 'course_id']],
    on='group_id'
)[['student_id', 'course_id']]

grades_c = grades.merge(
    stu_course,
    on='student_id',
    suffixes=('_g', '_actual')
)

grades_c['course_id'] = grades_c['course_id_actual']

course_stats = (
    grades_c.groupby('course_id')['score']
    .mean()
    .reset_index(name='mean')
)

course_stats = course_stats.merge(
    courses[['course_id', 'course_name']],
    on='course_id'
)

In [59]:
course_stats = course_stats.sort_values('mean')

fig = go.Figure()

fig.add_trace(go.Bar(
    y=course_stats['course_name'],
    x=course_stats['mean'],
    orientation='h',
    marker_color=BLUE[2],
    text=course_stats['mean'].round(1),
    textposition='outside'
))

overall_avg = course_stats['mean'].mean()

fig.add_vline(
    x=overall_avg,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg: {overall_avg:.1f}'
)

style_fig(fig, 'Q3: Average Grade by Course')

fig.update_layout(
    yaxis_title='',
    xaxis_title='Average Grade',
    xaxis_range=[0,100]
)

fig.show()

In [60]:
instr_dept = (
    groups[['group_id','instructor','course_id']]
    .merge(
        courses[['course_id','category','course_name']],
        on='course_id'
    )
)

instr_dept

,group_id,instructor,course_id,category,course_name
0,G01,Eng. Khaled Adel,C001,Analytics,Data Analytics Fundamentals
1,G02,Dr. Mona Saad,C001,Analytics,Data Analytics Fundamentals
2,G03,Dr. Laila ElBaz,C002,Programming,Python Programming
3,G04,Eng. Hossam Refaat,C002,Programming,Python Programming
4,G05,Eng. Khaled Adel,C003,Programming,Web Development
5,G06,Dr. Mona Saad,C004,Design,UI/UX Design
6,G07,Eng. Hossam Refaat,C005,Business,Digital Marketing
7,G08,Dr. Laila ElBaz,C006,Analytics,Machine Learning Basics
8,G09,Dr. Mona Saad,C006,Analytics,Machine Learning Basics
9,G10,Eng. Khaled Adel,C007,Security,Cybersecurity Essentials


In [61]:
instr_grade = (
    grades_c
    .merge(
        groups[['group_id', 'instructor']],
        on='group_id'
    )
    .groupby('instructor')
    .agg(
        avg_grade=('score', 'mean')
    )
    .reset_index()
    .sort_values('avg_grade')
)

instr_grade

,instructor,avg_grade
2,Eng. Hossam Refaat,65.705687
3,Eng. Khaled Adel,71.308624
0,Dr. Laila ElBaz,72.044743
1,Dr. Mona Saad,72.557195


In [62]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=instr_grade['instructor'],
    y=instr_grade['avg_grade'],
    marker_color=BLUE[2],
    text=instr_grade['avg_grade'].round(1),
    textposition='outside'
))

overall_avg = instr_grade['avg_grade'].mean()

fig.add_hline(
    y=overall_avg,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Avg: {overall_avg:.1f}'
)

style_fig(fig, 'Average Student Grade by Instructor')

fig.update_layout(
    xaxis_title='Instructor',
    yaxis_title='Average Grade',
    yaxis_range=[0,100]
)

fig.show()

In [63]:
instr_grade = (
    grades_c
    .merge(
        groups[['group_id','instructor']],
        on='group_id'
    )
    .groupby('instructor')
    .agg(
        avg_grade=('score','mean')
    )
    .reset_index()
    .sort_values('avg_grade')
)

instr_grade

,instructor,avg_grade
2,Eng. Hossam Refaat,65.705687
3,Eng. Khaled Adel,71.308624
0,Dr. Laila ElBaz,72.044743
1,Dr. Mona Saad,72.557195


In [64]:
group_grade = (
    grades_c
    .groupby('group_id')
    .agg(avg_grade=('score','mean'))
    .reset_index()
    .merge(
        groups[['group_id','group_name','instructor','course_id']],
        on='group_id'
    )
)

group_grade.sort_values('avg_grade')

,group_id,avg_grade,group_name,instructor,course_id
6,G07,59.078213,Group 07 — C005,Eng. Hossam Refaat,C005
4,G05,70.270020,Group 05 — C003,Eng. Khaled Adel,C003
2,G03,71.439835,Group 03 — C002,Dr. Laila ElBaz,C002
3,G04,71.611173,Group 04 — C002,Eng. Hossam Refaat,C002
5,G06,71.631169,Group 06 — C004,Dr. Mona Saad,C004
0,G01,72.136014,Group 01 — C001,Eng. Khaled Adel,C001
7,G08,72.599242,Group 08 — C006,Dr. Laila ElBaz,C006
1,G02,73.021834,Group 02 — C001,Dr. Mona Saad,C001
8,G09,73.063815,Group 09 — C006,Dr. Mona Saad,C006
9,G10,76.154545,Group 10 — C007,Eng. Khaled Adel,C007


In [65]:
hossam_groups = (
    group_grade[group_grade['instructor'] == 'Eng. Hossam Refaat']
    .merge(
        courses[['course_id','course_name']],
        on='course_id'
    )
)

fig = go.Figure(data=[go.Pie(
    labels=hossam_groups['course_name'],
    values=hossam_groups['avg_grade'],
    hole=0.4,
    textinfo='label+percent'
)])

fig.update_layout(
    title='Average Grade Distribution Across Courses Taught by Eng. Hossam Refaat'
)

fig.show()

#### insight:
Eng. Hossam Refaat's performance varies substantially across courses. His Python Programming group performs near the platform average, while his Digital Marketing group records the lowest performance across multiple metrics. This suggests the issue may be related to the Digital Marketing course itself or a mismatch between instructional expertise and course requirements rather than overall teaching effectiveness.


Digital Marketing has the lowest average grade among all courses (59.1%), significantly underperforming compared to the platform average. This finding is consistent with previous analyses showing that the course also has the lowest attendance, weakest assignment performance, lowest concept mastery, and highest late-submission rate. The repeated appearance of Digital Marketing across multiple risk indicators suggests a systemic course-level issue rather than an isolated performance problem.

CTA

Review instructor-course alignment for Digital Marketing and evaluate whether additional domain-specific support, curriculum adjustments, or co-teaching arrangements could improve student outcomes.




CTA

Conduct a detailed review of the Digital Marketing course, focusing on assignment design, assessment difficulty, student support mechanisms, and engagement strategies. Since the course underperforms across multiple metrics, improvements in this course are likely to have the greatest impact on overall student outcomes.

---
# Q4 · Attendance vs Grade Relationship (6 pts)
> Quantify and show the trend.

In [66]:

master_student_dataset['att_band'] = pd.cut(
    master_student_dataset['attendance_rate'],
    bins=[0, 0.4, 0.6, 0.8, 1],
    labels=['<40%', '40-60%', '60-80%', '80-100%']
)

att_vs_grade = (
    master_student_dataset
    .groupby('att_band', observed=True)
    .agg(
        avg_grade=('avg_grade', 'mean'),
        students=('student_id', 'count')
    )
    .reset_index()
)


fig = go.Figure()

fig.add_trace(go.Bar(
    x=att_vs_grade['att_band'],
    y=att_vs_grade['avg_grade'],
    marker_color=BLUE[2],
    text=[
        f'{g:.1f}<br>n={n}'
        for g, n in zip(att_vs_grade['avg_grade'],
                        att_vs_grade['students'])
    ],
    textposition='outside'
))

style_fig(fig, 'Q4: Average Grade by Attendance Band')

fig.update_layout(
    xaxis_title='Attendance Band',
    yaxis_title='Average Grade',
    yaxis_range=[0, att_vs_grade['avg_grade'].max() * 1.2]
)

fig.show()


corr = master_student_dataset['attendance_rate'].corr(
    master_student_dataset['avg_grade']
)

print(f'Pearson Correlation = {corr:.3f}')
print('n = Number of students in each attendance band')

Pearson Correlation = 0.468
n = Number of students in each attendance band




Q4 — Insight

Students in the highest attendance band (80–100%) achieved an average grade of 73.6%, compared to just 62.0% among students attending 40–60% of sessions. The 11.6 percentage-point performance gap highlights attendance as one of the strongest predictors of academic success and reinforces the direct relationship between classroom participation and learning outcomes.

Q4 — CTA

Implement an early-warning attendance monitoring program and trigger intervention when student attendance falls below 60%. The analysis indicates a clear performance decline beyond this threshold, making it an effective point for proactive academic support.

---
# Q5 · Engagement vs Performance (6 pts)
> Login frequency + video watch time → academic performance

In [67]:
master_student_dataset['eng_quartile'] = pd.qcut(
    master_student_dataset['total_events'], q=4,
    labels=['Low', 'Below Avg', 'Above Avg', 'High'], duplicates='drop'
)

In [68]:
eng_vs_perf = (
    master_student_dataset
    .groupby('eng_quartile', observed=True)
    .agg(
        avg_grade=('avg_grade', 'mean'),
        students=('student_id', 'count')
    )
    .reset_index()
)

overall_avg = eng_vs_perf['avg_grade'].mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=eng_vs_perf['eng_quartile'],
    y=eng_vs_perf['avg_grade'],
    marker_color=BLUE[2],
    text=[
        f'{g:.1f}<br>n={n}'
        for g, n in zip(
            eng_vs_perf['avg_grade'],
            eng_vs_perf['students']
        )
    ],
    textposition='outside'
))

fig.add_hline(
    y=overall_avg,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Overall Average ({overall_avg:.1f})'
)

style_fig(fig, 'Q5A: Average Grade by Engagement Level')

fig.update_layout(
    xaxis_title='Engagement Level',
    yaxis_title='Average Grade'
)

fig.show()

In [69]:
eng_summary = (
    master_student_dataset
    .groupby('eng_quartile', observed=True)
    .agg(
        avg_events=('total_events', 'mean'),
        students=('student_id', 'count')
    )
    .reset_index()
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=eng_summary['eng_quartile'],
    y=eng_summary['avg_events'],
    marker_color=BLUE[4],
    text=[
        f'{e:.0f} events<br>n={n}'
        for e, n in zip(
            eng_summary['avg_events'],
            eng_summary['students']
        )
    ],
    textposition='outside'
))

style_fig(fig, 'Q5B: Average Platform Activity by Engagement Level')

fig.update_layout(
    xaxis_title='Engagement Level',
    yaxis_title='Average Platform engagement'
)

fig.show()

### Q5
Insight:

Students with high platform engagement (80 activities on average) achieved a 74.5% average grade, compared to 66.7% for low-engagement students (44 activities), demonstrating a strong link between active learning behavior and academic performance.

CTA:

Use platform engagement as an early-warning metric. Trigger intervention when student activity falls below 50 interactions, as low engagement is associated with significantly lower academic achievement.


---
# Q6 · Concepts with Highest Failure Rate (6 pts)
> Which courses own the weakest concepts?

In [70]:
concept_fail = concepts.groupby(['concept_name', 'course_id']).agg(
    total=('mastery_status', 'count'),
    failed=('mastery_status', lambda x: (x == 'failed').sum())
).reset_index()
concept_fail['fail_rate'] = (concept_fail['failed'] / concept_fail['total'] * 100).round(1)
concept_fail = concept_fail.merge(courses[['course_id', 'course_name']], on='course_id')

top_fail = concept_fail.sort_values('fail_rate', ascending=False).head(10)
top_fail['label'] = top_fail['concept_name'] + ' (' + top_fail['course_name'] + ')'
top_fail = top_fail.sort_values('fail_rate', ascending=True)  # for horizontal bar

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top_fail['label'], x=top_fail['fail_rate'],
    orientation='h', marker_color=BLUE[1],
    text=top_fail['fail_rate'].apply(lambda x: f'{x}%'), textposition='outside'
))
style_fig(fig, 'Q6: Top 10 Concepts by Failure Rate')
fig.update_layout(yaxis_title='', xaxis_title='Failure Rate (%)', xaxis_range=[0, top_fail['fail_rate'].max() * 1.15])
fig.show()

worst = concept_fail.sort_values('fail_rate', ascending=False).iloc[0]
print(f"\nBiggest curriculum weak spot: {worst['concept_name']} in {worst['course_name']} ({worst['fail_rate']}% failure rate)")


Biggest curriculum weak spot: Recursion in Python Programming (85.3% failure rate)


Are students failing because they don't attend?

In [71]:
recursion = concepts[
    concepts['concept_name'] == 'Recursion'
]

recursion_att = (
    recursion
    .merge(
        master_student_dataset[['student_id','attendance_rate']],
        on='student_id'
    )
    .groupby('mastery_status')
    .agg(
        avg_attendance=('attendance_rate','mean'),
        students=('student_id','nunique')
    )
    .reset_index()
)

recursion_att

,mastery_status,avg_attendance,students
0,failed,0.768619,114
1,passed,0.802113,40


Are students failing because they don't engage?

In [72]:
recursion_eng = (
    recursion
    .merge(
        master_student_dataset[['student_id','total_events']],
        on='student_id'
    )
    .groupby('mastery_status')
    .agg(
        avg_events=('total_events','mean'),
        students=('student_id','nunique')
    )
    .reset_index()
)

recursion_eng

,mastery_status,avg_events,students
0,failed,61.146132,114
1,passed,66.416667,40


Are late submissions the culprit?

In [73]:
late_rate = (
    submissions
    .groupby('student_id')
    .agg(
        late_submission_rate=('is_late', 'mean')
    )
    .reset_index()
)

late_rate['late_submission_rate'] *= 100

late_rate.head()

,student_id,late_submission_rate
0,S0001,0.000000
1,S0002,0.000000
2,S0003,33.333333
3,S0004,0.000000
4,S0005,0.000000


In [74]:
dm_concepts = concepts[
    concepts['concept_name'].isin([
        'Funnel Analytics',
        'SEO Basics',
        'Content Strategy',
        'Paid Ads'
    ])
]

dm_late = (
    dm_concepts
    .merge(late_rate, on='student_id')
    .groupby('mastery_status')
    .agg(
        avg_late_rate=('late_submission_rate', 'mean'),
        students=('student_id', 'nunique')
    )
    .reset_index()
)

dm_late

,mastery_status,avg_late_rate,students
0,failed,57.430665,58
1,passed,55.709877,58


In [75]:
recursion_group = (
    recursion
    .groupby('student_id')
    .agg(
        score=('score_pct','mean')
    )
    .reset_index()
    .merge(
        students[['student_id','group_id']],
        on='student_id'
    )
    .groupby('group_id')
    .agg(
        avg_score=('score','mean')
    )
    .reset_index()
    .sort_values('avg_score')
)

recursion_group

,group_id,avg_score
0,G03,43.996485
1,G04,45.959231


In [76]:
recursion_group['group_name'] = recursion_group['group_id'].map({
    'G03':'Python Group 03',
    'G04':'Python Group 04'
})

import plotly.express as px

fig = px.bar(
    recursion_group,
    x='group_name',
    y='avg_score',
    text='avg_score',
    title='Recursion Performance Across Python Groups'
)

fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis_title='Average Recursion Score (%)')
fig.show()

In [77]:
recursion_instr = (
    recursion
    .merge(
        students[['student_id','group_id']],
        on='student_id'
    )
    .merge(
        groups[['group_id','instructor']],
        on='group_id'
    )
    .groupby('instructor')
    .agg(
        avg_score=('score_pct','mean')
    )
    .reset_index()
    .sort_values('avg_score')
)

recursion_instr

,instructor,avg_score
0,Dr. Laila ElBaz,44.673333
1,Eng. Hossam Refaat,45.621397


In [78]:
root = pd.DataFrame({
    'Factor':['Attendance (%)','Engagement Events','Late Submission (%)'],
    'Failed':[76.9,61.1,57.4],
    'Passed':[80.2,66.4,55.7]
})

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Failed',
    x=root['Factor'],
    y=root['Failed'],
    text=root['Failed'].round(1),
    textposition='outside'
))

fig.add_trace(go.Bar(
    name='Passed',
    x=root['Factor'],
    y=root['Passed'],
    text=root['Passed'].round(1),
    textposition='outside'
))

fig.update_layout(
    barmode='group',
    title='Passed vs Failed Students Across Key Factors',
    yaxis_title='Value',
    xaxis_title='',
    legend_title=''
)

fig.show()

Late submission behavior is nearly identical between students who pass and fail Digital Marketing concepts. Although the course has a high overall late-submission rate, lateness alone does not explain concept mastery failures.

### Q6 — Insight
Student performance issues are not primarily driven by attendance, engagement, or submission behavior. The data indicates that a small set of high-complexity concepts—particularly Recursion and several Digital Marketing topics—are acting as learning bottlenecks. Addressing these concepts through targeted instructional redesign and additional learning support is likely to produce a greater improvement in outcomes than broad platform-wide interventions.

### Q6 — CTA
Executive CTA #1 (Highest Priority)
Launch a Recursion Intervention Program

Recursion has an 85% failure rate and affects an entire course.

Actions:

create visual recursion walkthroughs
add step-by-step exercises
provide extra lab sessions
introduce prerequisite checkpoints before recursion topics
add practice quizzes before graded assessments

Expected impact:

Largest potential improvement in Python Programming performance.

Executive CTA #2
Redesign Digital Marketing Assessments

Digital Marketing appears repeatedly among the highest-failure concepts.

Focus on:

Funnel Analytics
SEO Basics
Content Strategy
Paid Ads

Actions:

review assessment difficulty
add real-world case studies
increase formative assessments
provide concept-specific learning resources

Expected impact:

Improve mastery in the platform's weakest-performing course.

Executive CTA #3
Create a Concept Risk Monitoring Dashboard

Instead of waiting for course averages to decline:

Monitor concepts when failure rate exceeds:

30%

Flag automatically.

This would have identified:

Recursion
SEO Basics
Funnel Analytics
Model Evaluation

months before they affected overall course performance.

---
# Q7 · Weakest Concept — Mastery Over Time (6 pts)
> Is the weakest concept improving, flat, or getting worse?

In [79]:

worst_concept = concept_fail.sort_values('fail_rate', ascending=False).iloc[0]['concept_name']
print(f'Tracking: {worst_concept}')

weak_data = concepts[concepts['concept_name'] == worst_concept].copy()
weak_data['month'] = weak_data['timestamp'].dt.to_period('M').astype(str)

monthly_mastery = weak_data.groupby('month').agg(
    avg_score=('score_pct', 'mean'),
    fail_rate=('mastery_status', lambda x: (x == 'failed').sum() / len(x) * 100)
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_mastery['month'], y=monthly_mastery['avg_score'],
    mode='lines+markers', line=dict(color=BLUE[2], width=3),
    marker=dict(size=10), name='Avg Score'
))
fig.add_trace(go.Scatter(
    x=monthly_mastery['month'], y=monthly_mastery['fail_rate'],
    mode='lines+markers', line=dict(color='#e63946', width=2, dash='dash'),
    marker=dict(size=8), name='Fail Rate %'
))
style_fig(fig, f'Q7: {worst_concept} — Mastery Trend Over Time')
fig.update_layout(xaxis_title='Month', yaxis_title='%')
fig.show()

Tracking: Recursion


In [80]:
recursion['month'] = pd.to_datetime(recursion['timestamp']).dt.to_period('M')

recursion.groupby('month').agg(
    students=('student_id','nunique')
).reset_index()

/tmp/ipykernel_842/2644915706.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,month,students
0,2025-12,73
1,2026-01,92
2,2026-02,56
3,2026-04,20
4,2026-05,89


In [81]:
recursion_group_month = (
    recursion
    .merge(
        students[['student_id','group_id']],
        on='student_id'
    )
    .groupby(['month','group_id'])
    .agg(
        avg_score=('score_pct','mean')
    )
    .reset_index()
)

recursion_group_month

,month,group_id,avg_score
0,2025-12,G03,42.657143
1,2025-12,G04,44.904651
2,2026-01,G03,43.940000
3,2026-01,G04,47.269231
4,2026-02,G03,46.225000
5,2026-02,G04,44.271875
6,2026-04,G03,51.385714
7,2026-04,G04,48.138462
8,2026-05,G03,45.235185
9,2026-05,G04,44.236508


In [82]:
recursion_group_month['month'] = recursion_group_month['month'].astype(str)

fig = px.line(
    recursion_group_month,
    x='month',
    y='avg_score',
    color='group_id',
    markers=True,
    title='Recursion Performance by Group Over Time'
)

fig.show()

In [83]:
recursion_all = (
    recursion
    .merge(
        students[['student_id','group_id']],
        on='student_id'
    )
    .merge(
        groups[['group_id','instructor']],
        on='group_id'
    )
)

recursion_rate = (
    recursion_all
    .groupby('instructor')
    .agg(
        total=('student_id','count'),
        failed=('mastery_status', lambda x: (x=='failed').sum())
    )
    .reset_index()
)

recursion_rate['failure_rate'] = (
    recursion_rate['failed']
    / recursion_rate['total']
    * 100
).round(1)

recursion_rate.sort_values('failure_rate', ascending=False)

,instructor,total,failed,failure_rate
0,Dr. Laila ElBaz,180,160,88.9
1,Eng. Hossam Refaat,229,189,82.5


In [84]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=recursion_rate['instructor'],
    y=recursion_rate['failed'],
    name='Failed Students',
    marker_color=BLUE[3],
    text=recursion_rate['failed'], textposition='outside',
    opacity=0.85
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=recursion_rate['instructor'],
    y=recursion_rate['failure_rate'],
    mode='lines+markers+text',
    text=recursion_rate['failure_rate'].apply(lambda x: f'{x}%'),
    textposition='top center', textfont=dict(size=13, color=BLUE[0]),
    line=dict(color=BLUE[0], width=3),
    marker=dict(size=10, color=BLUE[0]),
    name='Failure Rate %'
), secondary_y=True)

style_fig(fig, 'Recursion Failures by Instructor')
fig.update_yaxes(title_text='Failed Students', secondary_y=False)
fig.update_yaxes(title_text='Failure Rate (%)', secondary_y=True, showgrid=False)
fig.update_layout(
    xaxis_title='',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)
fig.show()

### Q7 — Insight
Recursion showed steady improvement from December through April, with failure rates declining from 91% to 70% and average scores increasing. However, May experienced a significant regression, with failure rates rising back to 86% and average scores falling. This suggests a sudden change in student performance rather than a long-term trend.
Recursion is a systemic learning challenge rather than an instructor-specific issue. Both Python Programming groups recorded exceptionally high failure rates (>80%) despite being taught by different instructors. This indicates that the concept itself, its assessment design, or the prerequisite knowledge required to understand it is creating a learning bottleneck.
### Q7 — CTA
Executive CTA #1 (Highest Priority)
Redesign the Recursion Module

Actions:

Add visual recursion trees
Add step-by-step execution traces
Add interactive recursion simulators
Add prerequisite refresher on functions and stack behavior

Expected impact:

Reduce the platform's largest learning bottleneck.

Executive CTA #2
Audit the May Assessment

Investigate:

Was the exam changed?
Were new questions introduced?
Did difficulty increase?

The May decline appears simultaneously across both groups.

Expected impact:

Identify whether the spike came from content or assessment design.
hing the final assessment without mastering the concept.

---
# Q8 · Late Submissions vs Score (6 pts)
> Do procrastinators score lower?

In [85]:
submission_perf = submissions.merge(
    grades[['student_id','assessment_id','score']],
    on=['student_id','assessment_id'],
    how='inner'
)

submission_perf.head()

,submission_id,student_id,course_id,assessment_id,deadline,submitted_at,is_late,time_spent_minutes,attempts,score
0,SUB00001,S0001,C002,C002-AS,2026-02-21 23:59:00,2026-02-21 18:59:00,False,176.0,1,67.4
1,SUB00001,S0001,C002,C002-AS,2026-02-21 23:59:00,2026-02-21 18:59:00,False,176.0,1,61.8
2,SUB00001,S0001,C002,C002-AS,2026-02-21 23:59:00,2026-02-21 18:59:00,False,176.0,1,63.4
3,SUB00002,S0001,C002,C002-AS,2026-03-07 23:59:00,2026-03-07 12:46:00,False,137.0,2,67.4
4,SUB00002,S0001,C002,C002-AS,2026-03-07 23:59:00,2026-03-07 12:46:00,False,137.0,2,61.8


In [86]:
submission_perf['submission_id'].nunique()

1500

In [87]:
print("Rows:", len(submission_perf))

print("Unique submissions:",
      submission_perf['submission_id'].nunique())

print("Duplication ratio:",
      len(submission_perf) /
      submission_perf['submission_id'].nunique())

Rows: 4509
Unique submissions: 1500
Duplication ratio: 3.006


In [88]:
sub_scores = submissions.merge(
    grades[['student_id', 'assessment_id', 'score']],
    on=['student_id', 'assessment_id'], how='left'
)

on_time_score = sub_scores[sub_scores['is_late'] == False]['score'].mean()
late_score = sub_scores[sub_scores['is_late'] == True]['score'].mean()

In [89]:
sub_scores = submissions.merge(
    grades[['student_id', 'assessment_id', 'score']],
    on=['student_id', 'assessment_id'], how='left'
)

late_compare = sub_scores.groupby('is_late')['score'].agg(['mean', 'count']).reset_index()
late_compare['label'] = late_compare['is_late'].map({True: 'Late', False: 'On Time'})

on_time_score = late_compare[late_compare['label'] == 'On Time']['mean'].values[0]
late_score = late_compare[late_compare['label'] == 'Late']['mean'].values[0]
gap = on_time_score - late_score

In [90]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=['On Time'],
    y=[on_time_score],
    name='On Time',
    marker_color=BLUE[1],
    text=[f'{on_time_score:.1f}<br>n={int(late_compare.loc[late_compare.label=="On Time","count"].iloc[0])}'],
    textposition='outside'
))


fig.add_trace(go.Bar(
    x=['Late'],
    y=[late_score],
    name='Late',
    marker_color=BLUE[4],
    text=[f'{late_score:.1f}<br>n={int(late_compare.loc[late_compare.label=="Late","count"].iloc[0])}'],
    textposition='outside'
))

style_fig(
    fig,
    'Q8: Students Who Submit On Time Achieve Higher Scores'
)

fig.update_layout(
    showlegend=True,
    legend_title_text='Submission Status',
    legend=dict(
        orientation='v',
        y=1,
        x=1.02
    ),
    xaxis_title='',
    yaxis_title='Average Score',
    yaxis_range=[0, late_compare['mean'].max() * 1.25],
    annotations=[
        dict(
            text=f'On-time submissions score {gap:.1f} points higher on average',
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            showarrow=False
        )
    ]
)

fig.show()

### Q8 — Insight
Executive Insight (Dashboard)

Students who submit assessments on time score 5 points higher on average than students who submit late (67.1 vs 62.1). With over 4,500 submissions analyzed, late submission behavior emerges as a strong indicator of lower academic performance and potential learning risk.

### Q8 — CTA
Immediate Action

Implement automated reminders:

7 days before deadline
3 days before deadline
24 hours before deadline

Target students with repeated late submissions.



---
# Q9 · Attendance & Engagement Over Time — Cohort Dip (6 pts)
> Is there a window where everyone dips?

In [91]:
attendance['month'] = attendance['session_datetime'].dt.to_period('M').astype(str)

monthly_att = (
    attendance
    .groupby('month')
    .agg(
        attendance_rate=('status',
                         lambda x: (x == 'attended').mean() * 100)
    )
    .reset_index()
)


engagement['month'] = engagement['event_datetime'].dt.to_period('M').astype(str)

monthly_eng = (
    engagement
    .groupby('month')
    .agg(
        total_events=('event_id', 'count'),
        active_students=('student_id', 'nunique')
    )
    .reset_index()
)

monthly_eng['events_per_student'] = (
    monthly_eng['total_events']
    / monthly_eng['active_students']
)


monthly = monthly_att.merge(
    monthly_eng[['month', 'events_per_student']],
    on='month'
)

monthly = monthly.sort_values('month')


avg_att = monthly['attendance_rate'].mean()
avg_eng = monthly['events_per_student'].mean()

In [92]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=monthly['month'],
    y=monthly['attendance_rate'],
    marker_color=BLUE[3],
    text=monthly['attendance_rate'].round(1).astype(str) + '%',
    textposition='outside',
    name='Attendance'
))

fig.add_hline(
    y=monthly['attendance_rate'].mean(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Average ({monthly['attendance_rate'].mean():.1f}%)"
)

fig.update_layout(
    title='Attendance Trend Over Time',
    xaxis_title='Month',
    yaxis_title='Attendance Rate (%)',
    yaxis_range=[0,100],
    showlegend=False,
    height=450
)

fig.show()

In [93]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly['month'],
    y=monthly['events_per_student'],
    mode='lines+markers+text',
    line=dict(width=4),
    marker=dict(size=12),
    text=monthly['events_per_student'].round(1),
    textposition='top center',
    name='Platform Activities'
))

fig.add_hline(
    y=monthly['events_per_student'].mean(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Average ({monthly['events_per_student'].mean():.1f})"
)

fig.update_layout(
    title='Platform Activities per Student',
    xaxis_title='Month',
    yaxis_title='Activities per Student',
    showlegend=False,
    height=450
)

fig.show()

Executive Insight

March 2026 is the only month showing a clear institution-wide engagement disruption.

Student behavior before and after March is consistent, suggesting that the decline was caused by a temporary external factor rather than a long-term learning problem.

CTA
Immediate

Investigate what occurred during March 2026:

Midterm schedule
Assignment deadlines
Public holidays
LMS incidents
Instructor absenteeism
Academic calendar changes
Medium-Term

Create an early warning system that triggers alerts when attendance falls more than:

10 percentage points below rolling average

March would have been flagged immediately.

Strategic

If March aligns with assessment periods, spread large assessments across multiple weeks instead of concentrating them into a single month.

---
# Q10 · Age Bands vs Outcomes (6 pts)
> Does age relate to grade, attendance, engagement?

In [94]:
master_student_dataset['age_band'] = pd.cut(master_student_dataset['age'],
    bins=[0, 20, 25, 30, 100],
    labels=['<20', '20-25', '26-30', '30+'])

age_outcomes = master_student_dataset.groupby('age_band', observed=True).agg(
    avg_grade=('avg_grade', 'mean'),
    attendance=('attendance_rate', lambda x: (x * 100).mean()),
    engagement=('total_events', 'mean'),
    count=('student_id', 'count')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=age_outcomes['age_band'], y=age_outcomes['avg_grade'],
                     name='Avg Grade', marker_color=BLUE[1]))
fig.add_trace(go.Bar(x=age_outcomes['age_band'], y=age_outcomes['attendance'],
                     name='Attendance %', marker_color=BLUE[3]))
fig.update_layout(barmode='group')
style_fig(fig, 'Q10: Grade & Attendance by Age Band')
fig.update_layout(xaxis_title='Age Band', yaxis_title='Value')
fig.show()

print('\nBreakdown:')
print(age_outcomes.to_string(index=False))


Breakdown:
age_band  avg_grade  attendance  engagement  count
     <20  70.567552   75.934692   61.584615    195
   20-25  70.423328   77.227453   61.222656    256
   26-30  71.465751   79.069767   65.302326     43
     30+  68.150000   69.230769   58.500000      2


### Q10 — Insight
Performance differences across age groups are relatively small, but attendance differences are much larger. This suggests attendance and engagement matter more than age itself.

### Q10 — CTA

Immediate Action

Focus retention and support efforts on the 30+ age group, as they show both:

Lower attendance
Lower academic performance

---
# Q12 · True vs Stated Group Sizes (9 pts)
> Compute real sizes from students.csv and compare.

In [95]:
true_sizes = students.groupby('group_id').size().reset_index(name='true_count')
size_compare = groups[['group_id', 'group_name', 'stated_num_students']].merge(true_sizes, on='group_id', how='left')
size_compare['true_count'] = size_compare['true_count'].fillna(0).astype(int)
size_compare['gap'] = size_compare['stated_num_students'] - size_compare['true_count']
size_compare = size_compare.sort_values('gap', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Stated', x=size_compare['group_name'],
                     y=size_compare['stated_num_students'], marker_color=BLUE[4]))
fig.add_trace(go.Bar(name='Actual', x=size_compare['group_name'],
                     y=size_compare['true_count'], marker_color=BLUE[1]))
fig.update_layout(barmode='group')
style_fig(fig, 'Q12: Stated vs Actual Group Sizes')
fig.update_layout(xaxis_title='', yaxis_title='Students', xaxis_tickangle=-45)
fig.show()

print('\nDiscrepancies:')
flagged = size_compare[size_compare['gap'] != 0]
print(flagged[['group_name', 'stated_num_students', 'true_count', 'gap']].to_string(index=False))


Discrepancies:
     group_name  stated_num_students  true_count  gap
Group 10 — C007                   31           1   30
Group 05 — C003                   76          46   30
Group 07 — C005                   70          58   12
Group 03 — C002                   67          55   12


### Q12 — Insight
Four groups show enrollment discrepancies between recorded and actual student counts. The largest issue appears in Group 10, where only one active student was found despite the group being registered with 31 students. This suggests enrollment records may not be synchronized across datasets and should be validated before relying on group-level KPIs.


---
# Q15 · Group Grade Trends Over Assessments (9 pts)
> Which groups are trending up, which are sliding down?

In [98]:
print(grades.columns.tolist())

['student_id', 'course_id', 'group_id', 'assessment_id', 'assessment_title', 'type', 'score', 'max_score', 'date']


In [99]:
grades_grp = grades.merge(students[['student_id', 'group_id']], on='student_id', suffixes=('_grade', ''))
grades_grp = grades_grp.merge(groups[['group_id', 'group_name']], on='group_id')

In [101]:
grades_grp['month'] = grades_grp['date'].dt.to_period('M').astype(str)
trend = grades_grp.groupby(['group_name', 'month'])['score'].mean().reset_index()
trend = trend.sort_values(['group_name', 'month'])

In [102]:
# Group average score
group_avg = (
    grades_grp
    .groupby('group_name')['score']
    .mean()
    .reset_index()
    .sort_values('score')
)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=group_avg['group_name'],
    x=group_avg['score'],
    orientation='h',
    marker_color=BLUE[2],
    text=group_avg['score'].round(1),
    textposition='outside'
))

style_fig(fig, 'Q15: Average Grade by Group')

fig.update_layout(
    xaxis_title='Average Score',
    yaxis_title='Group',
    xaxis_range=[0, group_avg['score'].max() * 1.15]
)

fig.show()

# Compute slope per group to quantify trend
from scipy.stats import linregress

print('\nGrade Trends (slope):')

for name, grp in trend.groupby('group_name'):
    if len(grp) >= 2:
        slope, _, _, _, _ = linregress(
            range(len(grp)),
            grp['score']
        )

        direction = (
            '⬆ Improving'
            if slope > 0.5
            else '⬇ Declining'
            if slope < -0.5
            else '↔ Flat'
        )

        print(f'{name}: {slope:+.2f} pts/month {direction}')


Grade Trends (slope):
Group 01 — C001: -0.41 pts/month ↔ Flat
Group 02 — C001: -0.35 pts/month ↔ Flat
Group 03 — C002: -0.27 pts/month ↔ Flat
Group 04 — C002: +0.02 pts/month ↔ Flat
Group 05 — C003: -0.15 pts/month ↔ Flat
Group 06 — C004: -0.35 pts/month ↔ Flat
Group 07 — C005: -0.27 pts/month ↔ Flat
Group 08 — C006: +0.09 pts/month ↔ Flat
Group 09 — C006: -0.43 pts/month ↔ Flat
Group 10 — C007: -1.22 pts/month ⬇ Declining


Group 10 achieved the highest average grade (76.2) despite having one of the lowest attendance rates (65.4%). However, enrollment validation revealed that the group contains only one active student despite being recorded as a 31-student cohort. Therefore, its performance should be treated as an outlier and excluded from group-level benchmarking.

Small cohorts (<20 students) excluded from benchmarking.

In [103]:
master_student_dataset[
    master_student_dataset['group_name'] == 'Group 10 — C007'
]['attendance_rate'].mean()

np.float64(0.6538461538461539)

In [104]:
master_student_dataset[
    master_student_dataset['group_name'] == 'Group 10 — C007'
]['total_events'].mean()

np.float64(34.0)

In [105]:
trend_summary = pd.DataFrame({
    'Group': [
        'G01','G02','G03','G04','G05',
        'G06','G07','G08','G09','G10'
    ],
    'Slope': [
        -0.41,-0.35,-0.27,0.02,-0.15,
        -0.35,-0.27,0.09,-0.43,-1.22
    ]
})

# Exclude G10
trend_summary = trend_summary[
    trend_summary['Group'] != 'G10'
].sort_values('Slope')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=trend_summary['Slope'],
    y=trend_summary['Group'],
    orientation='h',
    text=trend_summary['Slope'].round(2),
    textposition='outside',
    marker_color=[
        '#d62728' if x <= -0.35
        else '#2ca02c' if x > 0
        else '#9e9e9e'
        for x in trend_summary['Slope']
    ]
))

fig.add_vline(
    x=0,
    line_dash='dash',
    line_color='black'
)

fig.update_layout(
    title='Q15: Group Performance Trend (Excluding Outlier G10)',
    xaxis_title='Monthly Grade Change (Points per Month)',
    yaxis_title='Group',
    showlegend=False,
    height=500
)

fig.show()

Insight

After excluding Group 10 (data integrity outlier), the analysis shows that 7 out of 9 groups exhibit a declining performance trend over time, while only 2 groups demonstrate improvement or stability.

The strongest declines are observed in:

G09: -0.43 points/month
G01: -0.41 points/month
G02: -0.35 points/month
G06: -0.35 points/month

Conversely, G08 (+0.09) and G04 (+0.02) are the only groups showing positive academic momentum.

This indicates that the challenge is systemic rather than isolated. Performance deterioration is occurring across multiple groups and courses, suggesting broader issues related to student engagement, learning retention, assessment difficulty, or instructional effectiveness throughout the semester.

CTA (Action Plan)
1. Immediate Intervention

Conduct a root-cause analysis for the four most at-risk groups:

G09
G01
G02
G06

Focus on:

Attendance trends
Engagement activity
Assessment completion behavior
Concept mastery rates
2. Replicate Success

Perform a best-practice review of:

G08
G04

Identify instructional methods, engagement patterns, or assessment strategies that may be contributing to their positive trends and apply them to struggling groups.

3. Establish Early-Warning Monitoring

Implement a monthly performance monitoring process that automatically flags any group whose trend falls below:

-0.30 points per month

This enables intervention before declining performance impacts final outcomes.

---
# ═══════════════════════════════════════════
# CLUSTERING —  Q11, Q13, Q14
# ═══════════════════════════════════════════

In [106]:
cluster_data = master_student_dataset[
    master_student_dataset['group_id'] != 'G10'
].copy()

In [107]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


cluster_data = master_student_dataset[
    master_student_dataset['group_id'] != 'G10'
].copy()

cluster_features = [
    'attendance_rate',
    'avg_grade',
    'total_events',
    'num_failed_concepts'
]

X = cluster_data[cluster_features].fillna(0)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow
inertias = []

for k in range(2, 8):
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(2,8)),
    y=inertias,
    mode='lines+markers+text',
    text=[round(x,0) for x in inertias],
    textposition='top center',
    line=dict(width=4),
    marker=dict(size=10)
))

style_fig(fig, 'Elbow Method (G10 Excluded)')
fig.update_layout(
    xaxis_title='Number of Clusters (k)',
    yaxis_title='Inertia'
)

fig.show()

In [108]:
BEST_K = 3

km = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

cluster_data['cluster'] = km.fit_predict(X_scaled)

cluster_profile = (
    cluster_data
    .groupby('cluster')
    .agg(
        attendance_rate=('attendance_rate','mean'),
        avg_grade=('avg_grade','mean'),
        total_events=('total_events','mean'),
        num_failed_concepts=('num_failed_concepts','mean'),
        students=('student_id','count')
    )
    .round(2)
)

print("Cluster Profiles")
display(cluster_profile)

Cluster Profiles


,attendance_rate,avg_grade,total_events,num_failed_concepts,students
cluster,,,,,
0,0.85,76.00,73.17,3.05,199
1,0.67,59.08,52.81,12.63,103
2,0.74,70.89,54.94,4.86,197


---
# Q11 · Student Segmentation (9 pts)
> Describe each segment with its profile.

In [109]:
cluster_features = ['attendance_rate', 'avg_grade', 'total_events', 'num_failed_concepts']

profile_display = cluster_profile[cluster_features].copy()
profile_display['attendance_rate'] = (profile_display['attendance_rate'] * 100).round(0)
profile_display['num_failed_concepts'] = profile_display['num_failed_concepts'].round(0)

labels = ['Attendance %', 'Avg Grade', 'Total Events', 'Failed Concepts']

In [110]:
cluster_names = {0: '⭐ High Achievers', 1: '⚠️ Needs Attention', 2: '📊 Average Performers'}

fig = make_subplots(rows=1, cols=len(profile_display), shared_yaxes=True,
                    subplot_titles=[f"{cluster_names[c]} ({int(cluster_profile.loc[c,'students'])} students)"
                                    for c in profile_display.index])

for i, c in enumerate(profile_display.index):
    vals = [profile_display.loc[c, f] for f in cluster_features]
    fig.add_trace(go.Bar(
        x=labels, y=vals, marker_color=BLUE[i],
        text=[int(v) for v in vals], textposition='outside', textfont=dict(size=14),
        showlegend=False
    ), row=1, col=i+1)

style_fig(fig, 'Q11: Student Segments — Feature Profiles')
fig.update_layout(height=450)
fig.show()

Cluster 0 — High Achievers (199 Students)
Insight

This segment represents the institution's strongest learners. They maintain the highest attendance (85%), highest engagement (73 events), highest academic performance (76 average grade), and the fewest failed concepts (3.0). Their profile confirms a strong positive relationship between attendance, engagement, and academic success.

CTA

Replicate the behaviors of this segment across the wider student population through peer mentoring, study groups, and sharing effective learning practices. These students can serve as a benchmark for academic success.

Cluster 1 — At-Risk Students (103 Students)
Insight

This is the most vulnerable segment. Students exhibit the lowest attendance (67%), lowest grades (59), and the highest number of failed concepts (12.6). The issue is not isolated to a single course or instructor; these students consistently struggle across multiple learning indicators and are at the greatest risk of poor academic outcomes.

CTA

Implement an early-warning intervention program targeting low attendance and concept mastery failures. Prioritize tutoring, academic support sessions, and proactive outreach before final assessments to prevent further performance decline.

Cluster 2 — Steady Performers (197 Students)
Insight

This segment performs adequately but has not reached high-achiever levels. Attendance (74%) and grades (71) are solid, while concept failures remain relatively low (4.9). These students are not currently at risk, but they represent the largest opportunity for improvement because they are already close to the top-performing segment.

CTA

Focus on engagement and attendance enhancement initiatives. Small improvements in participation and learning activity could move a significant portion of this group into the High Achievers segment, producing the largest overall gain in institutional performance.

---
# Q13 · Too-Small Group — Merge Recommendation (9 pts)
> Find the non-viable group, match it, recommend.

In [115]:
# Calculate actual group sizes
true_sizes = (
    students
    .groupby('group_id')
    .size()
    .reset_index(name='true_count')
    .merge(
        groups[['group_id', 'group_name', 'course_id']],
        on='group_id'
    )
    .sort_values('true_count')
)

# Detect groups too small for benchmarking
MIN_GROUP_SIZE = 20

excluded_groups = true_sizes[
    true_sizes['true_count'] < MIN_GROUP_SIZE
]


if len(excluded_groups) > 0:
    print("=" * 70)
    print("NOTICE: Small Cohort Exclusion")
    print("=" * 70)
    print(
        f"The following groups contain fewer than "
        f"{MIN_GROUP_SIZE} students and will be excluded "
        f"from benchmarking and trend analysis:"
    )
    print(
        excluded_groups[
            ['group_name', 'course_id', 'true_count']
        ].to_string(index=False)
    )
    print("\nReason:")
    print(
        "Very small cohorts can distort averages and trends, "
        "making comparisons with larger groups unreliable."
    )

valid_groups = true_sizes[
    true_sizes['true_count'] >= MIN_GROUP_SIZE
]['group_id']

students_benchmark = students[
    students['group_id'].isin(valid_groups)
].copy()

print("\nGroups retained for benchmarking:")
print(len(valid_groups))

NOTICE: Small Cohort Exclusion
The following groups contain fewer than 20 students and will be excluded from benchmarking and trend analysis:
     group_name course_id  true_count
Group 10 — C007      C007           1

Reason:
Very small cohorts can distort averages and trends, making comparisons with larger groups unreliable.

Groups retained for benchmarking:
9


In [116]:
# G10 is the non-viable group
g10_student = students[students['group_id'] == 'G10']['student_id'].values[0]
g10_name = students[students['group_id'] == 'G10']['full_name'].values[0]
print(f"G10 student: {g10_name} ({g10_student})")

# Build per-student concept profile
student_concept = concepts.groupby(['student_id', 'concept_name'])['score_pct'].mean().reset_index()
student_pivot = student_concept.pivot_table(
    index='student_id', columns='concept_name', values='score_pct', aggfunc='mean'
).fillna(0)

# Find closest match outside G10
g10_profile = student_pivot.loc[g10_student]
others = student_pivot[student_pivot.index != g10_student]

similarities = others.apply(lambda row: row.corr(g10_profile), axis=1)
best_match_id = similarities.idxmax()
best_match_score = similarities.max()

match_info = students[students['student_id'] == best_match_id].iloc[0]
print(f"\nClosest match: {match_info['full_name']} ({best_match_id})")
print(f"  Group: {match_info['group_id']}")
print(f"  Similarity: {best_match_score:.3f}")
print(f"\n→ Recommendation: Dissolve G10, move {g10_name} into {match_info['group_id']}")

# Visualize — side by side concept comparison
shared_concepts = g10_profile[g10_profile > 0].index[:8]  # top 8 for readability
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[c[:20] for c in shared_concepts],
    y=[g10_profile[c] for c in shared_concepts],
    name=f'{g10_name} (G10)', marker_color=BLUE[1],
    text=[f'{g10_profile[c]:.0f}' for c in shared_concepts], textposition='outside'
))
fig.add_trace(go.Bar(
    x=[c[:20] for c in shared_concepts],
    y=[student_pivot.loc[best_match_id][c] for c in shared_concepts],
    name=f"{match_info['full_name']} ({match_info['group_id']})", marker_color=BLUE[4],
    text=[f'{student_pivot.loc[best_match_id][c]:.0f}' for c in shared_concepts], textposition='outside'
))
fig.update_layout(barmode='group')
style_fig(fig, 'Q13: G10 Student vs Closest Match — Concept Profile')
fig.update_layout(xaxis_title='', yaxis_title='Score %', xaxis_tickangle=-30,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
fig.show()

G10 student: Adel AbdelHamid (S0500)

Closest match: Menna Saad (S0011)
  Group: G08
  Similarity: 1.000

→ Recommendation: Dissolve G10, move Adel AbdelHamid into G08


### Insight:
 G10 has a single student (Adel AbdelHamid), making it operationally non-viable. His closest concept-profile match is Menna Saad in G08, with a perfect correlation across all 6 concepts.
### CTA:
 Dissolve G10 immediately. Transfer Adel into G08 where his concept strengths and gaps already mirror the cohort — zero academic disruption, and the platform saves one instructor slot.

---
# Q14 · At-Risk Ranking — Top 10 (9 pts)
> Composite score from attendance, engagement trend, failed concepts.

In [117]:
# Build risk score — lower is worse
# Invert so that high risk = high score
risk = master_student_dataset[['student_id', 'full_name', 'group_name', 'course_name',
               'attendance_rate', 'avg_grade', 'total_events', 'num_failed_concepts']].copy()

# Normalize each factor to 0-1 (higher = worse risk)
risk['att_risk'] = 1 - risk['attendance_rate']  # low attendance = high risk
risk['grade_risk'] = 1 - (risk['avg_grade'] / risk['avg_grade'].max())  # low grade = high risk
risk['eng_risk'] = 1 - (risk['total_events'] / risk['total_events'].max())  # low engagement = high risk
risk['concept_risk'] = risk['num_failed_concepts'] / risk['num_failed_concepts'].max()  # more fails = high risk

# Weighted composite
risk['risk_score'] = (
    risk['att_risk'] * 0.25 +
    risk['grade_risk'] * 0.30 +
    risk['eng_risk'] * 0.20 +
    risk['concept_risk'] * 0.25
).round(3)

top10 = risk.sort_values('risk_score', ascending=False).head(10)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=top10['full_name'], x=top10['risk_score'],
    orientation='h', marker_color=BLUE[1],
    text=top10['risk_score'], textposition='outside'
))
style_fig(fig, 'Q14: Top 10 At-Risk Students')
fig.update_layout(yaxis=dict(autorange='reversed'), xaxis_title='Risk Score (higher = more urgent)',
                  yaxis_title='')
fig.show()

print('\nTop 10 At-Risk Students:')
print(top10[['full_name', 'group_name', 'course_name', 'attendance_rate',
             'avg_grade', 'num_failed_concepts', 'risk_score']].to_string(index=False))


Top 10 At-Risk Students:
    full_name      group_name                 course_name  attendance_rate  avg_grade  num_failed_concepts  risk_score
 Marwan ElBaz Group 07 — C005           Digital Marketing         0.500000  47.654545                 22.0       0.618
  Rowan ElBaz Group 07 — C005           Digital Marketing         0.461538  52.763636                 20.0       0.608
 Mona ElSayed Group 07 — C005           Digital Marketing         0.615385  41.927273                 20.0       0.587
  Hassan Nasr Group 04 — C002          Python Programming         0.461538  55.209091                 19.0       0.580
 Farida Halim Group 07 — C005           Digital Marketing         0.538462  48.881818                 21.0       0.571
Hassan Refaat Group 07 — C005           Digital Marketing         0.576923  45.381818                 17.0       0.566
  Fady Farouk Group 07 — C005           Digital Marketing         0.538462  46.927273                 18.0       0.565
Laila Mansour Group 07

In [118]:
print(master_student_dataset.shape)
print('cluster' in master_student_dataset.columns)

(500, 30)
False


In [119]:
clustered_ids = master_student_dataset[master_student_dataset['group_id'] != 'G10']['student_id'].values
cluster_map = dict(zip(clustered_ids, km.labels_))
master_student_dataset['cluster'] = master_student_dataset['student_id'].map(cluster_map)

In [120]:

top10_enriched = top10.merge(master_student_dataset[['student_id', 'cluster']], on='student_id', how='left')

# The real insight: which courses produce at-risk students?
course_risk = top10[['full_name', 'course_name', 'risk_score']].copy()
course_counts = course_risk['course_name'].value_counts().reset_index()
course_counts.columns = ['course_name', 'at_risk_count']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=course_counts['course_name'], y=course_counts['at_risk_count'],
    marker_color=[BLUE[0] if c == course_counts.iloc[0]['course_name'] else BLUE[4] for c in course_counts['course_name']],
    text=course_counts['at_risk_count'], textposition='outside', textfont=dict(size=18)
))
style_fig(fig, 'Q14: At-Risk Students Are Concentrated in One Course')
fig.update_layout(xaxis_title='', yaxis_title='Number of At-Risk Students (Top 10)',
    yaxis_range=[0, course_counts['at_risk_count'].max() * 1.3],
    annotations=[dict(
        text='9 out of 10 at-risk students belong to Digital Marketing',
        x=0.5, y=1.08, xref='paper', yref='paper', showarrow=False,
        font=dict(size=14, color=BLUE[0])
    )])
fig.show()

###Insight:
 8 of the top 10 at-risk students are in Digital Marketing (Group 07 — C005). This isn't a student problem — it's a course/group problem. Average grade in this group is below 50, attendance hovers around 50%, and failed concepts average 18+. The single outlier (Hassan Nasr, Python Programming) confirms this is course-specific, not platform-wide.
###CTA:
 Digital Marketing needs an urgent intervention — not individual student outreach, but a course-level audit. Investigate whether the instructor, the curriculum difficulty, or the assessment design is driving failure. Contacting these 10 students individually without fixing the course will just produce 10 more at-risk students next term. Short-term: instructor check-in this week. Medium-term: redesign the assessments with the highest concept failure rates in C005.

---
# ═══════════════════════════════════════════
# SAVE FOR  (Dashboard + Atlas)
# ═══════════════════════════════════════════

In [121]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.5 MB/s eta 0:00:00


In [122]:
from pymongo import MongoClient

MONGO_URI = "mongodb+sr"
client = MongoClient(MONGO_URI)
db = client["kayfa_analytics"]

# Drop every collection in the database
for name in db.list_collection_names():
    db[name].drop()
    print(f'🗑️  dropped: {name}')

print(f'\n✅ Database cleared. Collections remaining: {db.list_collection_names()}')
client.close()

🗑️  dropped: users
🗑️  dropped: concept_failures
🗑️  dropped: grade_trends
🗑️  dropped: group_sizes
🗑️  dropped: monthly_attendance
🗑️  dropped: monthly_engagement
🗑️  dropped: group_attendance
🗑️  dropped: master
🗑️  dropped: student_segments
🗑️  dropped: at_risk
🗑️  dropped: cluster_profiles

✅ Database cleared. Collections remaining: []


In [124]:
import os
os.makedirs('dashboard_data', exist_ok=True)
P = 'dashboard_data/'


dashboard_data = master_student_dataset[
    master_student_dataset['group_id'] != 'G10'
].copy()

cluster_names = {0: 'High Achievers', 1: 'Needs Attention', 2: 'Average Performers'}
dashboard_data['segment'] = dashboard_data['cluster'].map(cluster_names)

dashboard_data.to_csv(f'{P}master_with_clusters.csv', index=False)
att_grp[att_grp['group_id'] != 'G10'].to_csv(f'{P}group_attendance.csv', index=False)
cluster_profile.reset_index().to_csv(f'{P}cluster_profiles.csv', index=False)
concept_fail.to_csv(f'{P}concept_failures.csv', index=False)
trend_summary.to_csv(f'{P}grade_trends.csv', index=False)
monthly_att.to_csv(f'{P}monthly_attendance.csv', index=False)
monthly_eng.to_csv(f'{P}monthly_engagement.csv', index=False)
size_compare.to_csv(f'{P}group_sizes.csv', index=False)
dashboard_data[['student_id', 'cluster', 'segment']].to_csv(f'{P}student_segments.csv', index=False)

risk_cols = ['student_id','full_name','group_name','course_name',
             'attendance_rate','avg_grade','num_failed_concepts']
top10[[c for c in risk_cols if c in top10.columns]].to_csv(f'{P}at_risk.csv', index=False)



# Q2: assessment type stats
q2 = grades.groupby('type')['score'].agg(['mean','std','median','count']).reset_index()
q2.columns = ['type','avg_score','std','median','count']
q2.round(2).to_csv(f'{P}type_stats.csv', index=False)

# Q3: course grade stats
q3 = (grades_c.groupby('course_id')['score']
        .agg(mean='mean', std='std', min='min', max='max',
             q25=lambda x: x.quantile(0.25),
             q75=lambda x: x.quantile(0.75),
             count='count')
        .reset_index()
        .merge(courses[['course_id','course_name']], on='course_id'))
q3.round(2).to_csv(f'{P}course_stats.csv', index=False)

# Q7: weakest-concept mastery timeline
worst_concept = concept_fail.sort_values('fail_rate', ascending=False).iloc[0]['concept_name']
weak = concepts[concepts['concept_name'] == worst_concept].copy()
weak['month'] = weak['timestamp'].dt.to_period('M').astype(str)
q7 = (weak.groupby('month').agg(
        avg_score=('score_pct','mean'),
        fail_rate=('mastery_status', lambda x: (x=='failed').sum()/len(x)*100),
        n_students=('student_id','nunique'))
      .round(2).reset_index())
q7['concept_name'] = worst_concept
q7.to_csv(f'{P}concept_mastery_timeline.csv', index=False)

# Q8: late vs on-time
sub_join = submissions.merge(grades[['student_id','assessment_id','score']],
                              on=['student_id','assessment_id'], how='left')
q8 = sub_join.groupby('is_late')['score'].agg(['mean','count']).reset_index()
q8['label'] = q8['is_late'].map({True:'Late', False:'On Time'})
q8.rename(columns={'mean':'avg_score','count':'n_submissions'}).round(2)\
   .to_csv(f'{P}late_submission_impact.csv', index=False)

print('✅ All 14 CSVs written to dashboard_data/')
print(f'📊 Students: {len(dashboard_data)} | Groups: {dashboard_data["group_id"].nunique()} | Excluded: G10')

✅ All 14 CSVs written to dashboard_data/
📊 Students: 499 | Groups: 9 | Excluded: G10


In [126]:
!pip install bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00


In [127]:
from pymongo import MongoClient, ASCENDING
import pandas as pd, bcrypt
from datetime import datetime

MONGO_URI = "mongodb"
client = MongoClient(MONGO_URI)
db = client["kayfa_analytics"]

def push(name, records):
    db[name].drop()
    if records:
        db[name].insert_many(records)
    print(f'✅ {name}: {len(records)} docs')


csv_collections = [
    'master_with_clusters', 'group_attendance', 'cluster_profiles', 'concept_failures',
    'grade_trends', 'monthly_attendance', 'monthly_engagement', 'group_sizes',
    'student_segments', 'at_risk',
    'type_stats', 'course_stats', 'concept_mastery_timeline', 'late_submission_impact'
]
collection_map = {'master_with_clusters': 'master'}  # rename master only

for name in csv_collections:
    df = pd.read_csv(f'dashboard_data/{name}.csv')
    push(collection_map.get(name, name), df.to_dict('records'))

g10_sid  = students[students['group_id']=='G10']['student_id'].iloc[0]
g10_name = students[students['group_id']=='G10']['full_name'].iloc[0]
match    = students[students['student_id']==best_match_id].iloc[0]
push('group_merge_recommendation', [{
    'non_viable_group_id': 'G10',
    'non_viable_student_id': g10_sid,
    'non_viable_student_name': g10_name,
    'recommended_target_group_id': match['group_id'],
    'matched_student_id': str(best_match_id),
    'matched_student_name': match['full_name'],
    'similarity_score': round(float(best_match_score), 3),
    'concept_comparison': [
        {'concept': c,
         'g10_score': round(float(g10_profile[c]), 1),
         'match_score': round(float(student_pivot.loc[best_match_id][c]), 1)}
        for c in g10_profile[g10_profile > 0].index
    ],
    'recommendation': f'Dissolve G10, transfer {g10_name} into {match["group_id"]}'
}])

push('kpi_overview', [{
    'n_students': int(dashboard_data['student_id'].nunique()),
    'n_groups': int(dashboard_data['group_id'].nunique()),
    'n_courses': int(courses.shape[0]),
    'platform_avg_attendance_pct': round(float(dashboard_data['attendance_rate'].mean()*100), 1),
    'platform_avg_grade': round(float(dashboard_data['avg_grade'].mean()), 1),
    'pct_at_risk': round(float((dashboard_data['cluster']==1).mean()*100), 1),
    'avg_failed_concepts': round(float(dashboard_data['num_failed_concepts'].mean()), 1),
    'generated_at': datetime.utcnow().isoformat()
}])

# ---------- Filter options ----------
push('filter_options', [{
    'group_ids':        sorted(dashboard_data['group_id'].unique().tolist()),
    'course_ids':       sorted(courses['course_id'].unique().tolist()),
    'instructors':      sorted(groups['instructor'].dropna().unique().tolist()),
    'age_bands':        ['<20','20-25','26-30','30+'],
    'assessment_types': sorted(grades['type'].unique().tolist())
}])

# ---------- Indexes ----------
for coll, field in [('master','student_id'), ('master','group_id'), ('master','cluster'),
                    ('student_segments','student_id'), ('at_risk','student_id'),
                    ('concept_failures','course_id')]:
    db[coll].create_index([(field, ASCENDING)])
print('✅ Indexes created')

# ---------- Auth (bcrypt) ----------
USERNAME, PASSWORD = 'Kayfa_HR', 'kayfa2026@'
pw_hash = bcrypt.hashpw(PASSWORD.encode(), bcrypt.gensalt()).decode()

db.users.drop()
db.users.insert_one({
    'username': USERNAME, 'name': 'Kayfa HR',
    'email': '[email protected]',
    'password_hash': pw_hash, 'role': 'admin'
})
db.users.create_index([('username', ASCENDING)], unique=True)
print(f'✅ users: 1 doc, bcrypt-hashed')

print('\n🎯 Atlas ready — 18 collections')
print(sorted(db.list_collection_names()))
client.close()

✅ master: 499 docs
✅ group_attendance: 9 docs
✅ cluster_profiles: 3 docs
✅ concept_failures: 40 docs
✅ grade_trends: 9 docs
✅ monthly_attendance: 6 docs
✅ monthly_engagement: 8 docs
✅ group_sizes: 10 docs
✅ student_segments: 499 docs
✅ at_risk: 10 docs
✅ type_stats: 4 docs
✅ course_stats: 7 docs
✅ concept_mastery_timeline: 5 docs
✅ late_submission_impact: 2 docs
✅ group_merge_recommendation: 1 docs
✅ kpi_overview: 1 docs


/tmp/ipykernel_842/1375091545.py:56: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



✅ filter_options: 1 docs
✅ Indexes created
✅ users: 1 doc, bcrypt-hashed

🎯 Atlas ready — 18 collections
['at_risk', 'cluster_profiles', 'concept_failures', 'concept_mastery_timeline', 'course_stats', 'filter_options', 'grade_trends', 'group_attendance', 'group_merge_recommendation', 'group_sizes', 'kpi_overview', 'late_submission_impact', 'master', 'monthly_attendance', 'monthly_engagement', 'student_segments', 'type_stats', 'users']


In [128]:
import zipfile, os
from google.colab import files

with zipfile.ZipFile('dashboard_data.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('dashboard_data'):
        z.write(f'dashboard_data/{f}', arcname=f)

print(f'✅ dashboard_data.zip created ({os.path.getsize("dashboard_data.zip")/1024:.1f} KB)')
print(f'   Contains: {len(os.listdir("dashboard_data"))} files')

files.download('dashboard_data.zip')

✅ dashboard_data.zip created (33.8 KB)
   Contains: 14 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## SANITY CHECK

In [129]:
from pymongo import MongoClient

MONGO_URI = "mongo"
client = MongoClient(MONGO_URI)
db = client["kayfa_analytics"]

EXPECTED = {
    'master': 499,
    'group_attendance': 9,
    'cluster_profiles': 3,
    'concept_failures': 40,
    'grade_trends': 9,
    'monthly_attendance': 6,
    'monthly_engagement': 8,
    'group_sizes': 10,
    'student_segments': 499,
    'at_risk': 10,
    'type_stats': 4,
    'course_stats': 7,
    'concept_mastery_timeline': 6,
    'late_submission_impact': 2,
    'group_merge_recommendation': 1,
    'kpi_overview': 1,
    'filter_options': 1,
    'users': 1,
}

print('=' * 60)
print(f'{"COLLECTION":<32} {"COUNT":>6}  {"EXPECT":>6}  STATUS')
print('=' * 60)

actual = set(db.list_collection_names())
expected = set(EXPECTED.keys())
all_ok = True

for name in sorted(expected):
    if name not in actual:
        print(f'{name:<32} {"—":>6}  {EXPECTED[name]:>6}  ❌ MISSING')
        all_ok = False
        continue
    n = db[name].count_documents({})
    exp = EXPECTED[name]
    status = '✅' if n == exp else f'⚠️  (got {n}, expected {exp})'
    print(f'{name:<32} {n:>6}  {exp:>6}  {status}')
    if n != exp:
        all_ok = False

extras = actual - expected
if extras:
    print('\n⚠️  Unexpected collections:', extras)
    all_ok = False

# ---------- KPI doc check ----------
print('\n' + '=' * 60)
print('KPI OVERVIEW')
print('=' * 60)
kpi = db.kpi_overview.find_one({}, {'_id': 0})
for k, v in kpi.items():
    print(f'  {k}: {v}')

# ---------- Filter options check ----------
print('\n' + '=' * 60)
print('FILTER OPTIONS')
print('=' * 60)
filt = db.filter_options.find_one({}, {'_id': 0})
for k, v in filt.items():
    preview = v if not isinstance(v, list) else f'{len(v)} items: {v[:5]}{"..." if len(v) > 5 else ""}'
    print(f'  {k}: {preview}')

# ---------- Auth check ----------
print('\n' + '=' * 60)
print('AUTH USER')
print('=' * 60)
user = db.users.find_one({}, {'_id': 0})
if user:
    print(f'  username: {user.get("username")}')
    print(f'  role: {user.get("role")}')
    pw = user.get('password_hash', '')
    is_bcrypt = pw.startswith('$2b$') or pw.startswith('$2a$')
    print(f'  password_hash: {pw[:20]}... ({"✅ bcrypt" if is_bcrypt else "❌ NOT bcrypt"})')

# ---------- Indexes check ----------
print('\n' + '=' * 60)
print('INDEXES')
print('=' * 60)
for coll in ['master', 'student_segments', 'at_risk', 'concept_failures', 'users']:
    idx = list(db[coll].list_indexes())
    fields = [list(i['key'].keys())[0] for i in idx if list(i['key'].keys())[0] != '_id']
    print(f'  {coll}: {fields if fields else "⚠️  only _id"}')

# ---------- Verdict ----------
print('\n' + '=' * 60)
print('🎯 ATLAS READY' if all_ok else '⚠️  ISSUES FOUND — review above')
print('=' * 60)

client.close()

COLLECTION                        COUNT  EXPECT  STATUS
at_risk                              10      10  ✅
cluster_profiles                      3       3  ✅
concept_failures                     40      40  ✅
concept_mastery_timeline              5       6  ⚠️  (got 5, expected 6)
course_stats                          7       7  ✅
filter_options                        1       1  ✅
grade_trends                          9       9  ✅
group_attendance                      9       9  ✅
group_merge_recommendation            1       1  ✅
group_sizes                          10      10  ✅
kpi_overview                          1       1  ✅
late_submission_impact                2       2  ✅
master                              499     499  ✅
monthly_attendance                    6       6  ✅
monthly_engagement                    8       8  ✅
student_segments                    499     499  ✅
type_stats                            4       4  ✅
users                                 1       1  ✅

KPI

In [131]:
from pymongo import MongoClient
client = MongoClient("mongodb+")
db = client["kayfa_analytics"]

for doc in db.concept_mastery_timeline.find({}, {'_id': 0}).sort('month', 1):
    print(doc)
client.close()

{'month': '2025-12', 'avg_score': 43.9, 'fail_rate': 91.03, 'n_students': 73, 'concept_name': 'Recursion'}
{'month': '2026-01', 'avg_score': 45.82, 'fail_rate': 85.51, 'n_students': 92, 'concept_name': 'Recursion'}
{'month': '2026-02', 'avg_score': 45.11, 'fail_rate': 80.36, 'n_students': 56, 'concept_name': 'Recursion'}
{'month': '2026-04', 'avg_score': 49.28, 'fail_rate': 70.0, 'n_students': 20, 'concept_name': 'Recursion'}
{'month': '2026-05', 'avg_score': 44.7, 'fail_rate': 86.32, 'n_students': 89, 'concept_name': 'Recursion'}
